# Class Imbalance Handling
Demonstrates SMOTE and undersampling strategies applied to the training split only.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.preprocessor import apply_smote, apply_undersampling
import warnings; warnings.filterwarnings('ignore')


In [ ]:
# Simulate imbalanced data for illustration (replace with real processed data)
np.random.seed(42)
n = 5000
X_demo = pd.DataFrame(np.random.randn(n, 10), columns=[f'f{i}' for i in range(10)])
y_demo = pd.Series(np.random.choice([0, 1], size=n, p=[0.91, 0.09]))

X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo, test_size=0.2, stratify=y_demo, random_state=42
)

print("Train set distribution BEFORE resampling:")
print(y_train.value_counts())


In [ ]:
# Apply SMOTE (preferred for e-commerce dataset)
X_smote, y_smote = apply_smote(X_train, y_train)
print("After SMOTE:")
print(y_smote.value_counts())


In [ ]:
# Apply Undersampling (preferred for creditcard — large dataset)
X_under, y_under = apply_undersampling(X_train, y_train, sampling_strategy=0.5)
print("After Undersampling (0.5 ratio):")
print(y_under.value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, y, title in [
    (axes[0], y_train, 'Before Resampling'),
    (axes[1], y_smote, 'After SMOTE'),
    (axes[2], y_under, 'After Undersampling'),
]:
    counts = y.value_counts()
    ax.bar(['Legit', 'Fraud'], [counts.get(0, 0), counts.get(1, 0)],
           color=['steelblue', 'tomato'])
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.suptitle('Resampling Strategy Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/resampling_comparison.png', bbox_inches='tight')
plt.show()


## Resampling Decision

| Dataset | Strategy | Justification |
|---|---|---|
| Fraud_Data.csv | **SMOTE** | Moderate size (~150k rows), 9% minority class. SMOTE generates synthetic samples in feature space preserving information; no data is discarded. |
| creditcard.csv | **Undersampling + SMOTE** | Extremely rare minority (0.17%, ~492 samples). Pure SMOTE on 492 samples risks overfitting synthetic data. Undersample majority to 10:1, then SMOTE to 2:1 for final training balance. |

**Critical rule**: Resampling is applied **only to the training set**. The test set retains the original class distribution to give realistic performance estimates.
